In [1]:
!huggingface-cli login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? (Y/n) n
Token is valid (permission: read).
The token `PC-Dell2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `PC-Dell2`


In [13]:
!apt update && apt install -y fonts-dejavu

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,683 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,725 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [4,363 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [4,517 kB]
Hit:13 https://ppa.launch

In [2]:
# ===================== Librerias necesarias ===================== #
import transformers
import torch
import gc
import os
import cv2
import numpy as np
import imageio
from PIL import Image, ImageDraw, ImageFont
from diffusers import DiffusionPipeline
from diffusers import PixArtAlphaPipeline

In [3]:
# ===================== Cargar modelo Llama3 ===================== #

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"

pipeline = transformers.pipeline(
  "text-generation",
  model=model_id,
  model_kwargs={"torch_dtype": torch.bfloat16},
  device_map="auto",
)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Device set to use cuda:0


In [4]:
# ===================== Arrays para guardar Prompts y Antiprompts ===================== #

prompts = []
antiprompts = []

phrases = [
    "Un dragón sin su jinete es una tragedia. Un jinete sin su dragón está muerto.",
    "Hasta el día de hoy, no he sido capaz de romper la conexión entre este chico, Peeta Mellark, el pan que me dio esperanza y el diente de león que me recordó que no estaba condenada",
    "Cuando se juega al juego de trono, solo se puede ganar o morir. No hay puntos intermedios."
]

In [5]:
# ===================== Generación de Prompts y Antiprompts ===================== #

for frase in phrases:
    print(f"\nFrase: {frase}")

    ## -------------------- Arrays para guardar los prompts que se van generando 1 a 1 -------------------- ##

    current_prompts = []
    current_antiprompts = []

    ## -------------------- Cantidad de prompts y antiprompts que se quieren crear -------------------- ##

    for _ in range(10):

        ## -------------------- Instrucciones para la generación de texto -------------------- ##

        phrase_prompts = [
            {"role": "user", "content": f"Genera solamente UN prompt que describa una escena inspirada en la frase para la generación de una imágen y UN antiprompt que describa qué cosas se deben evitar en la generación de la imagen, el formato debe ser 'Prompt: <frase generada>' lo mismo para el antiprompt, nada más que eso. en español. máximo de 77 tokens por cada uno. Describiendo una escena inspirada en la siguiente frase: '{frase}'"}
        ]

        phrase_prompts_outputs = pipeline(
            phrase_prompts,
            max_new_tokens=400,
            do_sample=True,
            temperature=0.6,
            top_p=0.9,
        )

        ## -------------------- Obtener texto generado -------------------- ##

        generated_text = [message["content"] for message in phrase_prompts_outputs[0]["generated_text"] if message["role"] == "assistant"][0]
        lines = generated_text.split("\n")

        ## -------------------- Guardar texto en su correspondiente array -------------------- ##

        for line in lines:
          if (line.startswith("Prompt") or line.startswith("**Prompt")):
            current_prompts.append(line)
          if (line.startswith("Antiprompt") or line.startswith("**Antiprompt")):
            current_antiprompts.append(line)

    ## -------------------- Juntar todos los prompts y antiprompts generados -------------------- ##

    prompts.append(current_prompts)
    antiprompts.append(current_antiprompts)

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Frase: Un dragón sin su jinete es una tragedia. Un jinete sin su dragón está muerto.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Frase: Hasta el día de hoy, no he sido capaz de romper la conexión entre este chico, Peeta Mellark, el pan que me dio esperanza y el diente de león que me recordó que no estaba condenada


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.



Frase: Cuando se juega al juego de trono, solo se puede ganar o morir. No hay puntos intermedios.


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


In [6]:
# ===================== Mostrar cada Prompt y Antiprompts generado para cada frase ===================== #
for i, (frase, prompt, antiprompt) in enumerate(zip(phrases, prompts, antiprompts), 1):
    print(f"Frase {i}:\n{frase}")

    for j, (prompt_, antiprompt_) in enumerate(zip(prompt, antiprompt), 1):
        print(f"{j} .- {prompt_}")
        print(f"{j} .- {antiprompt_}")

    print("-" * 50)


Frase 1:
Un dragón sin su jinete es una tragedia. Un jinete sin su dragón está muerto.
1 .- Prompt: Un dragón anciano y cansado yace sobre una colina cubierta de hierba seca, su escama brillante y sucia, mientras que en el fondo, un jinete caído y sin vida yace en el suelo, su armadura oxidada y rota.
1 .- Antiprompt: Evita generar una imagen con un dragón demasiado colorido o brillante, evita mostrar al jinete con expresiones de dolor o sufrimiento, evita incluir elementos sobrenaturales o fantasiosos que no estén directamente relacionados con la frase.
2 .- **Prompt:** Un campo de batalla abandonado, con un dragón grande y poderoso, solo y desolado, rodeado de cadáveres de jinetes y caballos, con un fondo de cielo oscuro y nubes negras.
2 .- **Antiprompt:** Evita mostrar un dragón con un jinete a su lado, no incluyas detalles de la batalla, no muestres un dragón atacando a un jinete, no incluyas texto o leyendas en la imagen.
3 .- Prompt: Un campo de batalla cubierto de niebla, donde

In [7]:
# ===================== Liberar espacio en GPU ===================== #

del pipeline
gc.collect()
torch.cuda.empty_cache()

In [8]:
# ===================== Agregar frase a la imagen ===================== #

def agregar_texto_y_crear_gif(image, frase, output_dir_prompt, gif_images, j, font_path, font_size, font_color, background_color):
    img_cv = np.array(image)
    img_cv = cv2.cvtColor(img_cv, cv2.COLOR_RGB2BGR)

    pil_image = Image.fromarray(img_cv)
    draw = ImageDraw.Draw(pil_image)
    font = ImageFont.truetype(font_path, font_size)

    max_width = pil_image.width - 40
    words = frase.split()
    lines = []
    current_line = ""
    for word in words:
        test_line = f"{current_line} {word}".strip()
        if draw.textlength(test_line, font=font) <= max_width:
            current_line = test_line
        else:
            lines.append(current_line)
            current_line = word
    lines.append(current_line)

    total_text_height = len(lines) * (font_size + 10)
    text_y = pil_image.height - total_text_height - 20

    background_padding = 20
    draw.rectangle(
        (10, text_y - background_padding, pil_image.width - 10, pil_image.height - 10),
        fill=background_color
    )

    y_offset = text_y
    for line in lines:
        draw.text((20, y_offset), line, font=font, fill=font_color)
        y_offset += font_size + 10

    img_cv_with_text = np.array(pil_image)

    image_path = os.path.join(output_dir_prompt, f"imagen_{j}.png")
    cv2.imwrite(image_path, img_cv_with_text)

    gif_images.append(pil_image)

    return gif_images

In [9]:
# ===================== Generar imágenes según modelo ===================== #

def Generar_imagenes(output_dir, gif_dir, phrases, prompts, antiprompts, modelo, modelo_uso, carpeta, guidence_scale, num_inference_steps, width, height, sampling_method):
  font_path = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
  font_size = 32
  font_color = (255, 255, 255)
  background_color = (0, 0, 0)

  for i, (frase, prompt, antiprompt) in enumerate(zip(phrases, prompts, antiprompts), 1):
    output_dir_prompt = os.path.join(output_dir, f"Frase_{i}", carpeta)
    os.makedirs(output_dir_prompt, exist_ok=True)

    gif_images = []

    for j, (prompt_, antiprompt_) in enumerate(zip(prompt, antiprompt), 1):
        print(f"Generando imagen para la frase {i}, oración {j}...")
        image = modelo_uso(prompt=prompt_, negative_prompt=antiprompt_, guidence_scale=guidence_scale, num_inference_steps=num_inference_steps, width=width, height=height, sampling_method=sampling_method).images[0]

        gif_images = agregar_texto_y_crear_gif(
            image=image,
            frase=frase,
            output_dir_prompt=output_dir_prompt,
            gif_images=gif_images,
            j=j,
            font_path=font_path,
            font_size=font_size,
            font_color=font_color,
            background_color=background_color
        )

    output_dir_gif = os.path.join(gif_dir, carpeta)
    os.makedirs(output_dir_gif, exist_ok=True)

    gif_path = os.path.join(output_dir_gif, f"Frase_{i}.gif")

    gif_images[0].save(gif_path, save_all=True, append_images=gif_images[1:], duration=500, loop=0)

In [10]:
# ===================== Cargar Modelo stabilityai/stable-diffusion-xl-base-1.0 ===================== #

Stabilityai = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16, use_safetensors=True, variant="fp16")
Stabilityai.to('cuda')

model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


scheduler_config.json:   0%|          | 0.00/479 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

diffusion_pytorch_model.fp16.safetensors:   0%|          | 0.00/167M [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

StableDiffusionXLPipeline {
  "_class_name": "StableDiffusionXLPipeline",
  "_diffusers_version": "0.33.1",
  "_name_or_path": "stabilityai/stable-diffusion-xl-base-1.0",
  "feature_extractor": [
    null,
    null
  ],
  "force_zeros_for_empty_prompt": true,
  "image_encoder": [
    null,
    null
  ],
  "scheduler": [
    "diffusers",
    "EulerDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "text_encoder_2": [
    "transformers",
    "CLIPTextModelWithProjection"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "tokenizer_2": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

In [14]:
# =====================  Valores por defecto del modelo ===================== #

output_dir = "images"
os.makedirs(output_dir, exist_ok=True)

gif_dir = "gif"
os.makedirs(gif_dir, exist_ok=True)

carpeta = "Stabilityai"
modelo = "Stabilityai"
guidence_scale = 5.0
num_inference_steps = 50
width = 1024
height = 1024
sampling_method = "ddim"

Generar_imagenes(output_dir, gif_dir, phrases, prompts, antiprompts, modelo, Stabilityai, carpeta, guidence_scale, num_inference_steps, width, height, sampling_method)

Generando imagen para la frase 1, oración 1...


  0%|          | 0/50 [00:00<?, ?it/s]

Generando imagen para la frase 1, oración 2...


  0%|          | 0/50 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (90 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['scena con tonos dorados y rojizos.']
Token indices sequence length is longer than the specified maximum sequence length for this model (90 > 77). Running this sequence through the model will result in indexing errors
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['scena con tonos dorados y rojizos.']


...

Generando imagen para la frase 3, oración 9...


  0%|          | 0/50 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['. la cuestión es : ¿ quién saldrá vivo de esta partida?"']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['. la cuestión es : ¿ quién saldrá vivo de esta partida?"']


Generando imagen para la frase 3, oración 10...


  0%|          | 0/50 [00:00<?, ?it/s]

In [17]:
# =====================  Valores modificados del modelo ===================== #
output_dir = "images"
os.makedirs(output_dir, exist_ok=True)

gif_dir = "gif"
os.makedirs(gif_dir, exist_ok=True)

carpeta = "Stabilityai - Modificado"
modelo = "Stabilityai"
guidence_scale = 10.0
num_inference_steps = 75
width = 1920
height = 1080
sampling_method = "pndm"

Generar_imagenes(output_dir, gif_dir, phrases, prompts, antiprompts, modelo, Stabilityai, carpeta, guidence_scale, num_inference_steps, width, height, sampling_method)

Generando imagen para la frase 1, oración 1...


  0%|          | 0/75 [00:00<?, ?it/s]

Generando imagen para la frase 1, oración 2...


  0%|          | 0/75 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['scena con tonos dorados y rojizos.']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['scena con tonos dorados y rojizos.']


...

Generando imagen para la frase 3, oración 9...


  0%|          | 0/75 [00:00<?, ?it/s]

The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['. la cuestión es : ¿ quién saldrá vivo de esta partida?"']
The following part of your input was truncated because CLIP can only handle sequences up to 77 tokens: ['. la cuestión es : ¿ quién saldrá vivo de esta partida?"']


Generando imagen para la frase 3, oración 10...


  0%|          | 0/75 [00:00<?, ?it/s]

In [18]:
# =====================  Liberar espacio de GPU ===================== #

del Stabilityai
gc.collect()
torch.cuda.empty_cache()

In [19]:
# ===================== Cargar Modelo PixArt-alpha/PixArt-XL-2-1024-MS ===================== #

PixArt = PixArtAlphaPipeline.from_pretrained("PixArt-alpha/PixArt-XL-2-1024-MS", torch_dtype=torch.float16)
PixArt = PixArt.to("cuda")

model_index.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.99G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/9.06G [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/719 [00:00<?, ?B/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some weights of the model checkpoint at /root/.cache/huggingface/hub/models--PixArt-alpha--PixArt-XL-2-1024-MS/snapshots/b89adadeccd9ead2adcb9fa2825d3fabec48d404/transformer were not used when initializing PixArtTransformer2DModel: 
 ['caption_projection.y_embedding']


In [20]:
# =====================  Valores por defecto del modelo ===================== #

output_dir = "images"
os.makedirs(output_dir, exist_ok=True)

gif_dir = "gif"
os.makedirs(gif_dir, exist_ok=True)

carpeta = "PixArt"
modelo = "PixArt"
guidence_scale = 4.5
num_inference_steps = 100
width = 1024
height = 1024
sampling_method = "ddim"

Generar_imagenes(output_dir, gif_dir, phrases, prompts, antiprompts, modelo, PixArt, carpeta, guidence_scale, num_inference_steps, width, height, sampling_method)


Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


Generando imagen para la frase 1, oración 1...


  0%|          | 0/100 [00:00<?, ?it/s]


Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


Generando imagen para la frase 1, oración 2...


...

Generando imagen para la frase 3, oración 9...


  0%|          | 0/100 [00:00<?, ?it/s]


Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...
The following part of your input was truncated because T5 can only handle sequences up to 120 tokens: ['vivo de esta partida?"']

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


Generando imagen para la frase 3, oración 10...


  0%|          | 0/100 [00:00<?, ?it/s]

In [21]:
# =====================  Valores modificados del modelo ===================== #

output_dir = "images"
os.makedirs(output_dir, exist_ok=True)

gif_dir = "gif"
os.makedirs(gif_dir, exist_ok=True)

carpeta = "PixArt - Modificado"
modelo = "PixArt"
guidence_scale = 10.0
num_inference_steps = 75
width = 1920
height = 1080
sampling_method = "pndm"

Generar_imagenes(output_dir, gif_dir, phrases, prompts, antiprompts, modelo, PixArt, carpeta, guidence_scale, num_inference_steps, width, height, sampling_method)


Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


Generando imagen para la frase 1, oración 1...


  0%|          | 0/75 [00:00<?, ?it/s]


Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


Generando imagen para la frase 1, oración 2...


...

Generando imagen para la frase 3, oración 9...


  0%|          | 0/75 [00:00<?, ?it/s]


Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...
The following part of your input was truncated because T5 can only handle sequences up to 120 tokens: ['vivo de esta partida?"']

Setting `clean_caption=True` requires the ftfy library but it was not found in your environment. Checkout the instructions on the
installation section: https://github.com/rspeer/python-ftfy/tree/master#installing and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.

Setting `clean_caption` to False...


Generando imagen para la frase 3, oración 10...


  0%|          | 0/75 [00:00<?, ?it/s]

In [22]:
import shutil
from google.colab import files
import os

images = "/content/images"
gif = "/content/gif"

images_zip = "/content/images.zip"
gif_zip = "/content/gif.zip"

shutil.make_archive(images_zip.replace(".zip", ""), 'zip', images)
shutil.make_archive(gif_zip.replace(".zip", ""), 'zip', gif)

files.download(images_zip)
files.download(gif_zip)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>